# Part 4 -- Conditioning justification language on what parts 1-3 already measured

Parts 1-3 established: vote accuracy ~0.41 for all three Gemma models; on
"dissociated" games (human crowd's modal target is wrong) models put ~2x more
vote mass on the crowd's wrong pick than on the true Werewolf; a surrogate
model predicts the LLM's vote from `werewolf_count` (accusations received)
and `claims_werewolf` (literal Werewolf self-claim) at 0.48-0.56 top-1
(random 0.23); and a DiMLex discourse-marker analysis found 31B uses ~4x the
Temporal-marker density of 2B, while Contingency density is flat across
correct/wrong votes and runs inverse to model capability.

This notebook asks whether the *language* of the justifications tracks those
already-established facts about the *votes*:

1. **Markers x surrogate-predictability** -- is Contingency density different
   for votes the surrogate's two pillars would have predicted vs. not?
2. **Markers x game class** -- on dissociated games, does Contingency/Temporal
   density differ between herded votes (crowd's wrong pick) and independent
   votes (true Werewolf)?
3. **Rule-reference lexicon** -- do models differ in how often they name
   roles/mechanics, and does 31B reference night mechanics more?
4. **Mention-check** -- when a vote tracks the surrogate's two pillars, does
   the justification actually *name* the evidence?
5. **Ground-truth swap tracking** -- an *algorithmic* (not word-list-based)
   check of rule-based deduction: when a voted player's role actually changed
   (start role != end role), does the justification correctly name the
   change, verified against ground truth rather than counted from a lexicon?

## Conventions (apply throughout, stated once here)

- All marker/lexicon counts are normalized **per 100 words**.
- Aggregation is **per-instance -> per-game mean -> mean across games**
  (equal game weighting). This matches the *extension* section of the
  existing DiMLex notebook, not its main body's run-level
  mean +/- SD-across-3-runs convention -- a deliberate, documented choice
  since the two sections of that notebook disagree.
- Stochastic runs (`run_1/2/3`, `decoding=="stochastic"`) are pooled as an
  empirical distribution over 3 T=1 samples (up to 3 instances/game feed the
  per-game mean). Greedy (`run_label=="greedy_t0"`) is always reported
  separately, never pooled with stochastic.
- Every number ships with `n_games` and `n_instances`. Every contrast ships a
  5000-draw 95% percentile bootstrap CI (games as the resampling unit, seeded
  `default_rng(0)`). Any CI spanning 0 is reported but flagged
  `inconclusive_at_this_n`, never narrated as a real difference.
- Interpretable methods only. Marker/lexicon mentions are not proof of rule
  use or reasoning; the mention-check in Analysis 4 is not proof of causal
  faithfulness. Caveats are repeated inline per section.


In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Could not find repo root {repo_name!r} above {Path.cwd()}")
        current = current.parent


REPO_ROOT = find_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "analysis"
DATA_PROCESSED = REPO_ROOT / "data" / "processed" / "lai2023"

MODEL_FOLDER_PATTERNS = {
    "2B": "*gemma-4-E2B*",
    "4B": "*gemma-4-E4B*",
    "31B": "*gemma-4-31B*",
}
MODEL_ORDER = ["2B", "4B", "31B"]
DECODING_ORDER = ["Stochastic", "Greedy"]

VOTE_FILE_REL = Path("base") / "voting" / "prompt_v4" / "vote_stability" / "tables" / "llm_vote_file_level.csv"
VOTE_GAME_REL = Path("base") / "voting" / "prompt_v4" / "vote_stability" / "tables" / "llm_vote_game_level.csv"

CROSS_MODEL_FEATURES_CSV = ANALYSIS_ROOT / "cross_model" / "voting" / "prompt_v4" / "tables" / "cross_model_game_features.csv"
VILLAGE_DISPERSION_CSV = ANALYSIS_ROOT / "human_outcomes" / "prompt_v4" / "tables" / "village_vote_dispersion.csv"
ACC_TARGETS_ROOT = DATA_PROCESSED / "accusation_transcripts" / "acc_targets"
IC_FEATURES_CSV = DATA_PROCESSED / "identity_claim_transcripts" / "ic_targets" / "player_conflict_features.csv"
DIMLEX_ASSIGNMENTS_CSV = REPO_ROOT / "src" / "justification_analysis" / "dimlex_marker_assignments.csv"

OUTPUT_DIR = ANALYSIS_ROOT / "cross_model" / "voting" / "prompt_v4" / "justification_analysis" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT, ALPHA = 5000, 0.05
RNG_BOOT = np.random.default_rng(0)

for name, path in [
    ("CROSS_MODEL_FEATURES_CSV", CROSS_MODEL_FEATURES_CSV),
    ("VILLAGE_DISPERSION_CSV", VILLAGE_DISPERSION_CSV),
    ("IC_FEATURES_CSV", IC_FEATURES_CSV),
    ("DIMLEX_ASSIGNMENTS_CSV", DIMLEX_ASSIGNMENTS_CSV),
    ("ACC_TARGETS_ROOT", ACC_TARGETS_ROOT),
]:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at {path}")

print("REPO_ROOT:", REPO_ROOT)
print("All configured paths exist.")

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
All configured paths exist.


## Load per-model vote tables, derive base populations

`VOTES_TEXT` keeps every row with real justification text (`status !=
"failed_parse"`) -- used by Analysis 3, which only needs text, not a vote
target. `VOTES_TARGETED` additionally drops `circle_vote` rows
(`status == "player_vote"` only) -- used by Analyses 1, 2 and 4, which all
condition on a real `chosen_player_name`.

In [2]:
def find_model_csv(folder_pattern, relative_path):
    matches = [d / relative_path for d in ANALYSIS_ROOT.glob(folder_pattern) if (d / relative_path).exists()]
    if not matches:
        raise FileNotFoundError(f"No file found for pattern {folder_pattern!r} / {relative_path}")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple matches for {folder_pattern!r}: {matches}")
    return matches[0]


file_frames, game_frames = [], []
for model_name, pattern in MODEL_FOLDER_PATTERNS.items():
    f = pd.read_csv(find_model_csv(pattern, VOTE_FILE_REL))
    f["model"] = model_name
    file_frames.append(f)
    g = pd.read_csv(find_model_csv(pattern, VOTE_GAME_REL))
    g["model"] = model_name
    game_frames.append(g)

votes_raw = pd.concat(file_frames, ignore_index=True)
games_raw = pd.concat(game_frames, ignore_index=True)
print("votes_raw:", votes_raw.shape, " games_raw:", games_raw.shape)

votes_raw["decoding_group"] = np.where(votes_raw["decoding"].str.lower().eq("stochastic"), "Stochastic", "Greedy")
votes_raw["justification"] = votes_raw["justification"].fillna("")
votes_raw["n_words"] = votes_raw["justification"].str.split().str.len().clip(lower=1)
votes_raw["justification_id"] = np.arange(len(votes_raw))

status_counts = votes_raw.groupby(["model", "decoding_group", "status"]).size().unstack(fill_value=0)
print("\nRow counts by status (excluded from VOTES_TEXT/VOTES_TARGETED as noted):")
print(status_counts)

VOTES_TEXT = votes_raw[votes_raw["status"] != "failed_parse"].copy()
VOTES_TARGETED = votes_raw[votes_raw["status"] == "player_vote"].copy()
print(f"\nVOTES_TEXT: {len(VOTES_TEXT)} rows (drops failed_parse only)")
print(f"VOTES_TARGETED: {len(VOTES_TARGETED)} rows (drops failed_parse + circle_vote)")

votes_raw: (2292, 16)  games_raw: (573, 22)

Row counts by status (excluded from VOTES_TEXT/VOTES_TARGETED as noted):
status                circle_vote  failed_parse  player_vote
model decoding_group                                        
2B    Greedy                   48             0          143
      Stochastic              131             5          437
31B   Greedy                   24             0          167
      Stochastic               69             0          504
4B    Greedy                   18             0          173
      Stochastic               34             0          539

VOTES_TEXT: 2287 rows (drops failed_parse only)
VOTES_TARGETED: 1963 rows (drops failed_parse + circle_vote)


## Canonical keys for the human-side annotation files

`game_id` strings are byte-identical across `llm_vote_file_level.csv`,
`llm_vote_game_level.csv`, `cross_model_game_features.csv` and
`village_vote_dispersion.csv`, so those join directly on `game_id`. The
accusation-target JSONs and `player_conflict_features.csv` key on their own
`source`/`session`/`game` columns, spelled inconsistently (`#` vs space,
case), so they need the canonical key from notebook 3
(`3_vote_surrogate_model.ipynb`), reused verbatim.

In [3]:
def canonical_session(x):
    x = str(x).strip().replace("#", " ")
    return re.sub(r"\s+", " ", x).lower()


def canonical_game(x):
    m = re.search(r"\d+", str(x))
    return f"game{int(m.group())}" if m else str(x).strip().lower()


def ckey(source, session, game):
    return (str(source).strip(), canonical_session(session), canonical_game(game))

## Marker-matching machinery (shared by DiMLex and the new rule-reference lexicon)

Ported from the main pipeline of `dimlex_justification_analysis (3).ipynb`
(`make_phrase_pattern` + longest-match-wins overlap suppression), generalized
into a small reusable `build_lexicon` / `find_matches` / `rate_table` triplet
so the same machinery serves Analyses 1/2 (DiMLex categories) and Analysis 3
(the new rule-reference lexicon).

In [4]:
WORD_PATTERN = re.compile(r"\b[\w]+(?:['’\-][\w]+)*\b", flags=re.UNICODE)


def make_phrase_pattern(marker):
    tokens = marker.split()
    body = r"\s+".join(re.escape(token) for token in tokens)
    return re.compile(rf"(?<!\w){body}(?!\w)", flags=re.IGNORECASE)


def build_lexicon(entries):
    """entries: iterable of (phrase, category) pairs -> lexicon df sorted longest-marker-first."""
    df = pd.DataFrame(list(entries), columns=["marker", "category"])
    df["pattern"] = df["marker"].map(make_phrase_pattern)
    df["marker_length"] = df["marker"].str.len()
    return df.sort_values(["marker_length", "marker"], ascending=[False, True]).reset_index(drop=True)


def find_matches(text, lexicon_df):
    candidates = []
    for row in lexicon_df.itertuples(index=False):
        for m in row.pattern.finditer(text):
            candidates.append({"category": row.category, "start": m.start(), "end": m.end()})
    candidates.sort(key=lambda c: (-(c["end"] - c["start"]), c["start"]))
    accepted, occupied = [], []
    for c in candidates:
        if any(c["start"] < e and c["end"] > s for s, e in occupied):
            continue
        accepted.append(c)
        occupied.append((c["start"], c["end"]))
    return accepted


def rate_table(df, lexicon_df, text_col="justification"):
    """Returns df.copy() with n_<category> and rate_<category> (per 100 words) columns added."""
    cats = sorted(lexicon_df["category"].unique())
    counts = {c: [] for c in cats}
    for text in df[text_col]:
        tally = {c: 0 for c in cats}
        for m in find_matches(text, lexicon_df):
            tally[m["category"]] += 1
        for c in cats:
            counts[c].append(tally[c])
    out = df.copy()
    for c in cats:
        out[f"n_{c}"] = counts[c]
        out[f"rate_{c}"] = 100 * out[f"n_{c}"] / out["n_words"]
    return out


dimlex_df = pd.read_csv(DIMLEX_ASSIGNMENTS_CSV, encoding="utf-8-sig")
dimlex_entries = list(
    dimlex_df.loc[dimlex_df["use_for_matching"], ["marker", "assigned_category"]].itertuples(index=False, name=None)
)
DIMLEX_LEXICON = build_lexicon(dimlex_entries)
print(f"DiMLex matching lexicon: {len(DIMLEX_LEXICON)} markers across categories {sorted(DIMLEX_LEXICON['category'].unique())}")

V_DIMLEX = rate_table(VOTES_TARGETED, DIMLEX_LEXICON)
# "as" sensitivity check for Temporal, mirrored from the established DiMLex robustness check
_as_count = V_DIMLEX["justification"].str.lower().str.count(r"\bas\b")
V_DIMLEX["rate_Temporal_noas"] = 100 * (V_DIMLEX["n_Temporal"] - _as_count).clip(lower=0) / V_DIMLEX["n_words"]
print(f"V_DIMLEX (VOTES_TARGETED with DiMLex rates): {V_DIMLEX.shape}")

DiMLex matching lexicon: 136 markers across categories ['Comparison', 'Contingency', 'Expansion', 'Temporal']


V_DIMLEX (VOTES_TARGETED with DiMLex rates): (1963, 28)


## Aggregation and bootstrap helpers

`game_then_mean` implements the equal-game-weighting convention. `boot_diff`
is the unpaired percentile-bootstrap contrast (two different, only
partially-overlapping instance subsets, e.g. predictable vs. not). `boot_diff_paired`
is used only in Analysis 3, where all three models share the literal same 191
games, so pairing removes per-transcript confounds. `boot_mean` is a
single-arm CI for Analysis 4's mention proportions.

In [5]:
def game_then_mean(df, cols, group_keys=("model", "decoding_group")):
    keys = [*group_keys, "game_id"]
    return df.groupby(keys)[list(cols)].mean().groupby(list(group_keys)).mean()


def boot_diff(a, b, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """Unpaired 95% percentile CI for mean(a) - mean(b); a, b are per-game values, independently resampled."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    da = a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1)
    db = b[rng.integers(0, len(b), (n_boot, len(b)))].mean(1)
    d = da - db
    lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean() - b.mean()), float(lo), float(hi)


def boot_diff_paired(a, b, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """Paired 95% percentile CI for mean(a) - mean(b); a, b are per-game arrays aligned on the
    SAME game order (index i refers to the same game in both). Used only for Analysis 3's
    cross-model comparison, where every model is scored on the identical 191-game set."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    assert len(a) == len(b), "paired bootstrap requires aligned arrays"
    n = len(a)
    if n < 2:
        return np.nan, np.nan, np.nan
    idx = rng.integers(0, n, (n_boot, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean() - b.mean()), float(lo), float(hi)


def boot_mean(a, n_boot=N_BOOT, alpha=ALPHA, rng=RNG_BOOT):
    """95% percentile CI for a single per-game-weighted mean (Analysis 4 proportions)."""
    a = np.asarray(a, float)
    if len(a) < 2:
        return (float(a.mean()) if len(a) else np.nan), np.nan, np.nan
    boots = a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean()), float(lo), float(hi)


def contrast_table(df, value_col, split_col, label_true, label_false, group_keys=("model", "decoding_group")):
    """Unpaired two-group contrast: per-game mean first within each split, per-game-weighted,
    reported with n_games and n_instances for both arms."""
    per_game = (
        df.groupby([*group_keys, split_col, "game_id"])[value_col]
        .agg(value="mean", n_instances="size")
        .reset_index()
    )
    rows = []
    for gkey, grp in per_game.groupby(list(group_keys)):
        a = grp.loc[grp[split_col] == True, "value"].values
        b = grp.loc[grp[split_col] == False, "value"].values
        n_inst_a = int(grp.loc[grp[split_col] == True, "n_instances"].sum())
        n_inst_b = int(grp.loc[grp[split_col] == False, "n_instances"].sum())
        diff, lo, hi = boot_diff(a, b)
        gkey_tuple = gkey if isinstance(gkey, tuple) else (gkey,)
        rows.append({
            **dict(zip(group_keys, gkey_tuple)),
            f"n_games_{label_true}": len(a),
            f"n_games_{label_false}": len(b),
            f"n_instances_{label_true}": n_inst_a,
            f"n_instances_{label_false}": n_inst_b,
            f"mean_{label_true}": round(float(a.mean()), 3) if len(a) else np.nan,
            f"mean_{label_false}": round(float(b.mean()), 3) if len(b) else np.nan,
            "diff": round(diff, 3) if not np.isnan(diff) else np.nan,
            "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
            "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
            "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
            "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)


print("Every contrast below reports n_games and n_instances for both arms; any CI spanning 0")
print("is flagged inconclusive_at_this_n and is not narrated as a real difference in the text.")

Every contrast below reports n_games and n_instances for both arms; any CI spanning 0
is flagged inconclusive_at_this_n and is not narrated as a real difference in the text.


# Analysis 1 -- Markers x surrogate-predictability (proxy)

The surrogate model (`3_vote_surrogate_model.ipynb`) does not save per-instance
out-of-fold predictions, so this uses the necessary **proxy**: a vote is
"predictable" if its target is that game's most-accused player
(werewolf-type accusations only, matching the surrogate's actual feature --
deception-type accusations exist in the same data but are a separate,
unused feature there and are left out here too) **or** a Werewolf
self-claimer. This is not the surrogate's held-out score; it is a
necessity-based proxy built from the same two dialogue features. If
Contingency density is flat between predictable and unpredictable votes, that
is evidence reasoning language is decorative rather than tracking the actual
evidence driving the vote.

In [6]:
acc_rows = []
for p in sorted(ACC_TARGETS_ROOT.rglob("*.json")):
    try:
        rec = json.loads(p.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue
    meta = rec.get("metadata", {})
    if not all(meta.get(k) for k in ("source", "session", "game")):
        continue
    key = ckey(meta["source"], meta["session"], meta["game"])
    for item in rec.get("items", []):
        for rel in item.get("relations", []):
            if rel.get("type") != "werewolf":
                continue
            for pl in rel.get("accused", []):
                if pl != "UNKNOWN":
                    acc_rows.append({"key": key, "player": pl})

acc_df = pd.DataFrame(acc_rows)
werewolf_counts = (
    acc_df.groupby(["key", "player"]).size().rename("werewolf_count").reset_index()
    if not acc_df.empty else pd.DataFrame(columns=["key", "player", "werewolf_count"])
)
print(f"werewolf-type accusation events: {len(acc_df)}; games covered: {werewolf_counts['key'].nunique()}")

ic = pd.read_csv(IC_FEATURES_CSV)
ic["key"] = [ckey(s, se, g) for s, se, g in zip(ic["source"], ic["session"], ic["game"])]


def parse_roles(v):
    try:
        return set(json.loads(v)) if isinstance(v, str) else set()
    except json.JSONDecodeError:
        return set()


ic["claims_werewolf"] = ic["roles_claimed"].apply(lambda v: int("Werewolf" in parse_roles(v)))
ic_feats = ic[["key", "player", "claims_werewolf"]]
print(f"identity-claim rows: {len(ic_feats)}; games covered: {ic_feats['key'].nunique()}")

werewolf-type accusation events: 1184; games covered: 173
identity-claim rows: 632; games covered: 189


In [7]:
def name_map(players):
    return {p.strip().lower(): p for p in players}


roster_by_game = (
    games_raw.drop_duplicates(subset="game_id")
    .assign(players=lambda d: d["player_names"].map(json.loads),
            source=lambda d: d["source"], session_name=lambda d: d["session_name"], game_key=lambda d: d["game_key"])
)

candidate_rows = []
for row in roster_by_game.itertuples(index=False):
    game_id, players = row.game_id, row.players
    key = ckey(row.source, row.session_name, row.game_key)
    nmap = name_map(players)
    wc = {p: 0 for p in players}
    for r in werewolf_counts[werewolf_counts["key"] == key].itertuples(index=False):
        canon = nmap.get(str(r.player).strip().lower())
        if canon is not None:
            wc[canon] += int(r.werewolf_count)
    cw = {p: 0 for p in players}
    for r in ic_feats[ic_feats["key"] == key].itertuples(index=False):
        canon = nmap.get(str(r.player).strip().lower())
        if canon is not None:
            cw[canon] = max(cw[canon], int(r.claims_werewolf))
    for p in players:
        candidate_rows.append({"game_id": game_id, "player": p, "werewolf_count": wc[p], "claims_werewolf": cw[p]})

candidates = pd.DataFrame(candidate_rows)
print(f"candidate rows: {len(candidates)} over {candidates['game_id'].nunique()} games")


def most_accused(group):
    if group["werewolf_count"].max() == 0:
        return None  # undefined: no recorded accusation evidence in this game at all
    top = group["werewolf_count"].max()
    return sorted(group.loc[group["werewolf_count"] == top, "player"])[0]  # deterministic alphabetical tie-break


most_accused_by_game = candidates.groupby("game_id").apply(most_accused, include_groups=False).to_dict()
claims_werewolf_by_game = (
    candidates.groupby("game_id")
    .apply(lambda g: set(g.loc[g["claims_werewolf"] == 1, "player"]), include_groups=False)
    .to_dict()
)

n_defined = sum(1 for v in most_accused_by_game.values() if v is not None)
n_with_claim = sum(1 for v in claims_werewolf_by_game.values() if len(v) > 0)
print(f"games with most_accused_player defined: {n_defined} / {len(most_accused_by_game)}")
print(f"games with all-zero accusations (most_accused_player = None): {len(most_accused_by_game) - n_defined}")
print(f"games with >=1 Werewolf self-claimer: {n_with_claim} / {len(claims_werewolf_by_game)}")

candidate rows: 864 over 191 games


games with most_accused_player defined: 173 / 191
games with all-zero accusations (most_accused_player = None): 18
games with >=1 Werewolf self-claimer: 76 / 191


In [8]:
V1 = V_DIMLEX.copy()
V1["most_accused_player"] = V1["game_id"].map(most_accused_by_game)
V1["is_predictable"] = (
    (V1["chosen_player_name"] == V1["most_accused_player"])
    | V1.apply(lambda r: r["chosen_player_name"] in claims_werewolf_by_game.get(r["game_id"], set()), axis=1)
)

print(V1.groupby(["model", "decoding_group", "is_predictable"]).size().unstack(fill_value=0))

surrogate_means = (
    V1.groupby(["model", "decoding_group", "is_predictable", "game_id"])["rate_Contingency"]
    .mean().reset_index()
    .groupby(["model", "decoding_group", "is_predictable"])["rate_Contingency"].mean().unstack()
    .rename(columns={True: "mean_predictable", False: "mean_unpredictable"})
    .reset_index()
)
surrogate_contingency_ci = contrast_table(V1, "rate_Contingency", "is_predictable", "predictable", "unpredictable")
print("\nContingency density: predictable vs. unpredictable votes")
print(surrogate_contingency_ci)

is_predictable        False  True 
model decoding_group              
2B    Greedy             49     94
      Stochastic        153    284
31B   Greedy             77     90
      Stochastic        209    295
4B    Greedy             62    111
      Stochastic        214    325



Contingency density: predictable vs. unpredictable votes
  model decoding_group  n_games_predictable  n_games_unpredictable  n_instances_predictable  n_instances_unpredictable  mean_predictable  \
0    2B         Greedy                   94                     49                       94                         49             3.293   
1    2B     Stochastic                  126                     83                      284                        153             3.135   
2   31B         Greedy                   90                     77                       90                         77             2.435   
3   31B     Stochastic                  121                     94                      295                        209             2.689   
4    4B         Greedy                  111                     62                      111                         62             2.326   
5    4B     Stochastic                  134                     99                      325           

### Appendix -- patch to save true out-of-fold predictions (not implemented here)

In `notebooks/03_llm_voting_outcome_analysis/3_vote_surrogate_model.ipynb`'s
`cv_evaluate`, inside the `for fold in range(N_FOLDS):` loop, after
`te["score"] = model.predict_proba(X_te)[:, 1]`, accumulate
`te[["key", "instance", "player", "label", "score"]].assign(method=method, seed=seed, fold=fold)`
into a list defined before the seed loop. After both loops finish,
`pd.concat(oof_rows, ignore_index=True)` and write it to
`OUTPUT_DIR / "surrogate_oof_predictions.csv"`. `train_df["instance"]` is a
`(key, run_label)` tuple, so the true out-of-fold grain would be
`(game key, run_label, player)` -- one row per candidate per fold per seed
per method. A future Analysis 1 could then replace the proxy `is_predictable`
above with the actual OOF top-1 pick (`score == score.max()` within
`instance`), averaged across the 3 CV seeds, instead of the accusation/claim
proxy used here.

# Analysis 2 -- Markers x game class (herded vs. independent)

Restricted to `dissociated` games (the human crowd's modal target is
disjoint from the true Werewolf set -- reusing the exact `game_class`
already computed in `2_cross_model_vote_comparison.ipynb`, not recomputed
here). Within a dissociated game, a vote instance is **independent** if its
target is the true Werewolf, **herded** if its target is the crowd's wrong
modal pick, or **other** (targeted a real player who is neither -- excluded
from the headline contrast, counted separately). By construction of
`dissociated`, herded and independent never overlap for a single instance.

In [9]:
cmf = pd.read_csv(CROSS_MODEL_FEATURES_CSV)
dissociated_games = cmf.loc[cmf["game_class"] == "dissociated", ["model", "game_id"]].drop_duplicates()
print(f"dissociated (model, game_id) pairs: {len(dissociated_games)}")

village = pd.read_csv(VILLAGE_DISPERSION_CSV)
village["village_top_target_names"] = village["village_top_target_names"].apply(json.loads)
village_targets_by_game = village.set_index("game_id")["village_top_target_names"].apply(set).to_dict()

wolves_by_model_game = (
    games_raw.assign(wolves=lambda d: d["correct_player_names"].map(lambda v: set(json.loads(v))))
    .set_index(["model", "game_id"])["wolves"].to_dict()
)

V2 = (
    V_DIMLEX.merge(dissociated_games, on=["model", "game_id"], how="inner")
)


def classify_instance(row):
    wolves = wolves_by_model_game.get((row["model"], row["game_id"]), set())
    village_top = village_targets_by_game.get(row["game_id"], set())
    if row["chosen_player_name"] in wolves:
        return "independent"
    if row["chosen_player_name"] in village_top:
        return "herded"
    return "other"


V2["vote_class"] = V2.apply(classify_instance, axis=1)
vote_class_counts = V2.groupby(["model", "decoding_group", "vote_class"]).size().unstack(fill_value=0)
print(vote_class_counts)

excluded_other = V2[V2["vote_class"] == "other"].groupby(["model", "decoding_group"]).size().rename("n_excluded_other").reset_index()

V2_contrast = V2[V2["vote_class"] != "other"].copy()
V2_contrast["is_herded"] = V2_contrast["vote_class"].eq("herded")

herded_contingency_ci = contrast_table(V2_contrast, "rate_Contingency", "is_herded", "herded", "independent")
herded_contingency_ci.insert(2, "marker_category", "Contingency")
herded_temporal_ci = contrast_table(V2_contrast, "rate_Temporal", "is_herded", "herded", "independent")
herded_temporal_ci.insert(2, "marker_category", "Temporal")
markers_by_herded_independent_ci = pd.concat([herded_contingency_ci, herded_temporal_ci], ignore_index=True)
print("\nContingency/Temporal density: herded vs. independent votes (dissociated games only)")
print(markers_by_herded_independent_ci)

dissociated (model, game_id) pairs: 141


vote_class            herded  independent  other
model decoding_group                            
2B    Greedy              17           12     11
      Stochastic          57           35     24
31B   Greedy              19           12     11
      Stochastic          67           26     36
4B    Greedy              19           11     15
      Stochastic          58           36     44



Contingency/Temporal density: herded vs. independent votes (dissociated games only)
   model decoding_group marker_category  n_games_herded  n_games_independent  n_instances_herded  n_instances_independent  mean_herded  \
0     2B         Greedy     Contingency              17                   12                  17                       12        3.399   
1     2B     Stochastic     Contingency              28                   19                  57                       35        3.042   
2    31B         Greedy     Contingency              19                   12                  19                       12        2.430   
3    31B     Stochastic     Contingency              29                   15                  67                       26        2.736   
4     4B         Greedy     Contingency              19                   11                  19                       11        2.063   
5     4B     Stochastic     Contingency              25                   17           

# Analysis 3 -- Rule-reference lexicon

**Judgment call, flagged for review before treating results as final.** The
role vocabulary is derived dynamically from the union of `start_roles`/
`end_roles` across all 191 games (not hardcoded), minus two slot values that
are not role *names*:

- `"Moderator"` -- a transcript-slot artifact (the human who ran the game),
  appearing in only 3/191 games.
- `"Center Card"` -- a real ONUW end-state (the Drunk blind-swaps, so their
  final card is an unidentified center card; it is the vote target's end role
  in 6 of ~1963 targeted votes) but it names a *location*, not an identity.
  Leaving it in `role_name` would also let longest-match-wins score the 12
  literal "center card" mentions in justification text as a role, stealing
  them from `"center"` in `card_mechanic` where a location reference belongs.
  Moving it shifts ~0.01 per 100 words between the two categories -- far
  smaller than any contrast reported below, but corrected for cleanliness.

The mechanic-term list below is
a starting proposal, split into `card_mechanic` and `night_order_mechanic` so
the "31B references night mechanics more" hypothesis isn't diluted by
generic card-handling terms. `night` is flagged as the noisiest inclusion
(it can catch non-mechanical uses like "last night narrating events"); a
`_noNight` sensitivity variant is computed alongside the raw rate, mirroring
the established DiMLex `_noas` robustness check.

### Important limitation on `night_order_mechanic` -- do not narrate it

Two independent problems make this category unfit to carry the "31B
references night mechanics more" hypothesis:

1. **Construct validity.** The models only ever see the public transcript;
   they have no access to night information. So a justification saying
   "Erin said she looked at the center" is the model *relaying another
   player's public claim*, not the model reasoning from night-order
   knowledge. The category cannot distinguish those two things, so a higher
   rate does not license a claim about rule understanding.
2. **Sparsity.** See the coverage diagnostic below -- once the polysemous
   bare `night` is excluded, this category is backed by roughly 5% of
   justifications (about 15 documents for 2B). Bootstrap CIs resample games,
   not matches, so a CI here can exclude 0 while resting on a handful of
   documents.

`card_mechanic` is the better-motivated test of the same underlying idea:
`swapped`/`robbed`/`center` are the *publicly discussed consequences* of
night actions, they carry no privileged-information problem, and they are
backed by ~44% of justifications. Read the hypothesis off that category,
and treat `night_order_mechanic` as exploratory only.

An earlier draft of this list also included `peek`/`peeked`/`peeking`, which
matched **zero** times across all 2287 justifications -- guessed vocabulary
the models do not use. They have been removed; the coverage table below is
the check that catches this class of mistake.

**A further caveat on `card_mechanic`:** `rob`/`robbed`/`robbing` and the
removed `peek*` terms do not appear anywhere in the actual rules text shown
to the models (`src/prompts/onuw_rules_v2.txt`), which says the Robber
"swaps their card with another player's card" -- never "steals" or "robs".
"rob(bed)" is analyst vocabulary describing the mechanic, not rules
vocabulary the models were given, kept here only because it is a common
paraphrase, not because it is grounded in the prompt.

**Limitation on `night_order_mechanic` -- do not narrate this category as a
finding.** Two independent problems make it unfit to test "31B reasons about
night mechanics more":

1. *Construct validity.* The models only see the public transcript; they
   have no access to night information. A justification saying "Erin said
   she looked at the center" is the model *relaying another player's public
   claim*, not demonstrating night-order deduction. A word count cannot
   distinguish that from a model actually *using* the rule as a premise
   ("since the Tanner does not wake up at night, Julie is lying").
2. *Sparsity.* Per the coverage table below, this category is backed by
   ~12% of justifications, 5% once the polysemous bare `night` is excluded --
   about 15 documents for 2B. A bootstrap CI can exclude 0 while resting on a
   handful of documents, because the resampling unit is games, not matches.

`card_mechanic` (44% coverage) is the better-motivated test of the same idea
-- `swapped`/`robbed`/`center` are the publicly-discussed *consequences* of
night actions, with no privileged-information problem. Read the "31B
references rules more" finding off `role_name` and `card_mechanic`; treat
`night_order_mechanic` as exploratory only. **Analysis 5 below is the
ground-truth-anchored replacement for the deduction question this category
cannot answer.**

In [10]:
def role_vocab(games_df):
    roles = set()
    for col in ("start_roles", "end_roles"):
        for v in games_df[col].dropna():
            roles.update(json.loads(v))
    return roles


# "Moderator": transcript-slot artifact (the human who ran the game), not an ONUW role.
# "Center Card": a real end-state (the Drunk blind-swaps, so their final card is an
#   unidentified center card) but NOT a role *name* -- it names a location, not an identity.
#   It is the vote target's end role in only 6 of ~1963 targeted votes. Critically, leaving it
#   in role_name would let longest-match-wins score the 12 literal "center card" mentions in
#   justification text as a ROLE rather than letting "center" (card_mechanic) catch them,
#   which is where a location reference belongs.
NON_ROLE_SLOT_VALUES = {"Moderator", "Center Card"}
ALL_ROLES = role_vocab(games_raw) - NON_ROLE_SLOT_VALUES
print(f"Role-slot values excluded as non-roles: {sorted(NON_ROLE_SLOT_VALUES)}")
print("Derived role vocabulary:")
print(sorted(ALL_ROLES))
role_entries = [(role, "role_name") for role in ALL_ROLES]

# JUDGMENT CALL -- review before treating Analysis 3 results as final.
CARD_MECHANIC_TERMS = [
    "center",                        # the 3 unused cards in the middle -- core ONUW concept
    "swap", "swapped", "swapping",   # Troublemaker/Robber card-swap mechanic
    "rob", "robbed", "robbing",      # Robber's specific action, distinct from generic "swap"
    "no werewolf",                   # the circle-vote outcome phrase
]
# NOTE: "peek"/"peeked"/"peeking" were in an earlier draft and matched ZERO times across all
# 2287 justifications -- guessed vocabulary the models simply do not use. Removed.
NIGHT_ORDER_MECHANIC_TERMS = [
    "wake up", "woke up",            # night-phase framing language
    "look at", "looked at",          # looking-at-a-card phrasing (the verb the models actually use)
    "night",                         # cheapest catch-all for night-order references;
                                      # noisiest inclusion -- see the _noNight sensitivity variant below
]
mechanic_entries = (
    [(t, "card_mechanic") for t in CARD_MECHANIC_TERMS]
    + [(t, "night_order_mechanic") for t in NIGHT_ORDER_MECHANIC_TERMS]
)

RULE_LEXICON = build_lexicon(role_entries + mechanic_entries)
print(f"\nRule-reference lexicon: {len(RULE_LEXICON)} terms across {sorted(RULE_LEXICON['category'].unique())}")

Role-slot values excluded as non-roles: ['Center Card', 'Moderator']
Derived role vocabulary:
['Doppelganger', 'Drunk', 'Hunter', 'Insomniac', 'Mason', 'Minion', 'Revealer', 'Robber', 'Seer', 'Tanner', 'Troublemaker', 'Villager', 'Werewolf']

Rule-reference lexicon: 26 terms across ['card_mechanic', 'night_order_mechanic', 'role_name']


In [11]:
J3 = rate_table(VOTES_TEXT, RULE_LEXICON)
_night_count = J3["justification"].str.lower().str.count(r"\bnight\b")
J3["n_night_order_mechanic_noNight"] = (J3["n_night_order_mechanic"] - _night_count).clip(lower=0)
J3["rate_night_order_mechanic_noNight"] = 100 * J3["n_night_order_mechanic_noNight"] / J3["n_words"]

RATE_COLS_3 = ["rate_role_name", "rate_card_mechanic", "rate_night_order_mechanic", "rate_night_order_mechanic_noNight"]
rule_rates_by_model = game_then_mean(J3, RATE_COLS_3).round(3)
print(rule_rates_by_model)

                      rate_role_name  rate_card_mechanic  rate_night_order_mechanic  rate_night_order_mechanic_noNight
model decoding_group                                                                                                  
2B    Greedy                   5.605               0.849                      0.158                              0.016
      Stochastic               5.630               0.699                      0.143                              0.035
31B   Greedy                   8.453               1.701                      0.292                              0.132
      Stochastic               8.363               1.823                      0.310                              0.145
4B    Greedy                   4.308               0.371                      0.122                              0.057
      Stochastic               4.456               0.363                      0.107                              0.043


### Coverage diagnostic -- how much text actually backs each category

A per-100-words rate says nothing about how many documents contribute to it.
A category matched by a handful of justifications can still produce a
bootstrap CI that excludes 0, because the resampling unit is the game, not the
match. The table below reports total occurrences and the share of
justifications containing at least one term, so sparse categories can be
discounted on sight rather than read as equals of the dense ones.

In [12]:
coverage_rows = []
for cat in ["role_name", "card_mechanic", "night_order_mechanic", "night_order_mechanic_noNight"]:
    n_col = f"n_{cat}"
    occurrences = int(J3[n_col].sum())
    with_any = int(J3[n_col].gt(0).sum())
    coverage_rows.append({
        "category": cat,
        "total_occurrences": occurrences,
        "justifications_with_ge1": with_any,
        "n_justifications": len(J3),
        "coverage_pct": round(100 * with_any / len(J3), 1),
    })
rule_lexicon_coverage = pd.DataFrame(coverage_rows)
print(rule_lexicon_coverage.to_string(index=False))

SPARSE_THRESHOLD_PCT = 20.0
sparse = rule_lexicon_coverage.loc[
    rule_lexicon_coverage["coverage_pct"] < SPARSE_THRESHOLD_PCT, "category"
].tolist()
print(f"\nCategories below {SPARSE_THRESHOLD_PCT:.0f}% coverage (do NOT narrate as findings): {sparse}")

                    category  total_occurrences  justifications_with_ge1  n_justifications  coverage_pct
                   role_name              10312                     2273              2287          99.4
               card_mechanic               1658                     1003              2287          43.9
        night_order_mechanic                317                      274              2287          12.0
night_order_mechanic_noNight                124                      115              2287           5.0

Categories below 20% coverage (do NOT narrate as findings): ['night_order_mechanic', 'night_order_mechanic_noNight']


In [13]:
def paired_cross_model_ci(df, value_col, decoding_group, model_pairs=(("31B", "2B"), ("31B", "4B"), ("4B", "2B"))):
    per_game = (
        df[df["decoding_group"] == decoding_group]
        .groupby(["model", "game_id"])[value_col].mean().unstack("model")
    )
    per_game = per_game.dropna()
    rows = []
    for m1, m2 in model_pairs:
        diff, lo, hi = boot_diff_paired(per_game[m1].values, per_game[m2].values)
        rows.append({
            "decoding_group": decoding_group, "model_a": m1, "model_b": m2,
            "n_games": len(per_game),
            "mean_a": round(per_game[m1].mean(), 3), "mean_b": round(per_game[m2].mean(), 3),
            "diff": round(diff, 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3),
            "excludes_zero": bool(lo > 0 or hi < 0),
            "inconclusive_at_this_n": bool(not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)


rule_ci_rows = []
for dg in DECODING_ORDER:
    for col in RATE_COLS_3:
        t = paired_cross_model_ci(J3, col, dg)
        t.insert(0, "marker_category", col)
        rule_ci_rows.append(t)
rule_reference_bootstrap_ci = pd.concat(rule_ci_rows, ignore_index=True)
print(rule_reference_bootstrap_ci)

                      marker_category decoding_group model_a model_b  n_games  mean_a  mean_b   diff  ci_low  ci_high  excludes_zero  \
0                      rate_role_name     Stochastic     31B      2B      191   8.363   5.630  2.733   2.413    3.050           True   
1                      rate_role_name     Stochastic     31B      4B      191   8.363   4.456  3.907   3.623    4.197           True   
2                      rate_role_name     Stochastic      4B      2B      191   4.456   5.630 -1.174  -1.415   -0.932           True   
3                  rate_card_mechanic     Stochastic     31B      2B      191   1.823   0.699  1.124   0.926    1.332           True   
4                  rate_card_mechanic     Stochastic     31B      4B      191   1.823   0.363  1.461   1.274    1.660           True   
5                  rate_card_mechanic     Stochastic      4B      2B      191   0.363   0.699 -0.337  -0.444   -0.233           True   
6           rate_night_order_mechanic     Stocha

# Analysis 4 -- Mention-check (faithfulness proxy)

**Caveat:** mentioning an entity or Werewolf-claim language is not proof the
justification's reasoning is causally faithful to that evidence -- it is the
cheapest available proxy, reporting only that the concept was named in the
text. Pillar 1 (accusation): among votes whose target is that game's
most-accused player, does the justification name that player? Pillar 2
(self-claim): among votes whose target self-claimed Werewolf, does the
justification mention the word "werewolf" at all? The two pillars are **not
mutually exclusive** (a target can satisfy both) -- each has an independent
denominator and the percentages should not be summed.

In [14]:
def name_mentioned(text, name):
    return re.search(rf"(?<!\w){re.escape(name)}(?!\w)", text, flags=re.IGNORECASE) is not None


WEREWOLF_WORD = re.compile(r"\bwerewolf\b", flags=re.IGNORECASE)


def werewolf_language_mentioned(text):
    return bool(WEREWOLF_WORD.search(text))


V4 = VOTES_TARGETED.copy()
V4["most_accused_player"] = V4["game_id"].map(most_accused_by_game)
V4["target_is_most_accused"] = V4["chosen_player_name"] == V4["most_accused_player"]
V4["target_claims_werewolf"] = V4.apply(
    lambda r: r["chosen_player_name"] in claims_werewolf_by_game.get(r["game_id"], set()), axis=1
)

pillar1 = V4[V4["target_is_most_accused"]].copy()
pillar1["mentions_target"] = pillar1.apply(lambda r: name_mentioned(r["justification"], r["chosen_player_name"]), axis=1)

pillar2 = V4[V4["target_claims_werewolf"]].copy()
pillar2["mentions_werewolf_language"] = pillar2["justification"].map(werewolf_language_mentioned)

print(f"Pillar 1 (accusation) instances: {len(pillar1)}   Pillar 2 (self-claim) instances: {len(pillar2)}")

Pillar 1 (accusation) instances: 857   Pillar 2 (self-claim) instances: 560


In [15]:
def mention_rate_table(df, mention_col, pillar_label):
    per_game = df.groupby(["model", "decoding_group", "game_id"])[mention_col].mean().reset_index()
    rows = []
    for (mdl, dg), grp in per_game.groupby(["model", "decoding_group"]):
        n_inst = int(df[(df["model"] == mdl) & (df["decoding_group"] == dg)].shape[0])
        mean, lo, hi = boot_mean(grp[mention_col].values)
        rows.append({
            "model": mdl, "decoding_group": dg, "pillar": pillar_label,
            "n_games": len(grp), "n_instances": n_inst,
            "pct_mentioning": round(100 * mean, 1) if not np.isnan(mean) else np.nan,
            "ci_low": round(100 * lo, 1) if not np.isnan(lo) else np.nan,
            "ci_high": round(100 * hi, 1) if not np.isnan(hi) else np.nan,
        })
    return pd.DataFrame(rows)


mention_check = pd.concat([
    mention_rate_table(pillar1, "mentions_target", "accusation_pillar"),
    mention_rate_table(pillar2, "mentions_werewolf_language", "self_claim_pillar"),
], ignore_index=True)
print(mention_check)

   model decoding_group             pillar  n_games  n_instances  pct_mentioning  ci_low  ci_high
0     2B         Greedy  accusation_pillar       66           66           100.0   100.0    100.0
1     2B     Stochastic  accusation_pillar       95          199            97.9    94.7    100.0
2    31B         Greedy  accusation_pillar       69           69           100.0   100.0    100.0
3    31B     Stochastic  accusation_pillar       96          225           100.0   100.0    100.0
4     4B         Greedy  accusation_pillar       78           78           100.0   100.0    100.0
5     4B     Stochastic  accusation_pillar      101          220           100.0   100.0    100.0
6     2B         Greedy  self_claim_pillar       45           45           100.0   100.0    100.0
7     2B     Stochastic  self_claim_pillar       57          136           100.0   100.0    100.0
8    31B         Greedy  self_claim_pillar       33           33           100.0   100.0    100.0
9    31B     Stochas

# Analysis 5 -- Ground-truth swap tracking (algorithmic deduction check)

The RQ needs an *algorithmic* way to establish rule-based deductive
reasoning, not a word list -- a lexicon can only measure vocabulary, never
correctness. This checks a **falsifiable claim** instead: when the voted
player's role actually changed (`voted_player_start_role !=
voted_player_end_role` -- already-existing ground-truth columns, no new
joins), does the justification correctly name the *current* (end) role?
Uses only the closed, externally-sourced 13-role vocabulary from Analysis 3
(`ALL_ROLES`) -- no analyst-invented phrases.

Mentioning **both** the start role and the end role for the target is the
target class: it is an explicit swap narration ("she started as Villager but
ended as Werewolf"), a stronger and more specific signal than matching the
end role alone, which could result from unrelated evidence (e.g. accusation
counts) without any tracked deduction.

**Caveats:** (1) co-occurrence is not proof of assertion -- the same
"cheapest available proxy" caveat as Analysis 4 applies. (2) This only
diagnoses votes where the target actually swapped; it says nothing about
non-swap votes. (3) The `neither` bucket (mentions some other role) is
reported but not interpreted without further qualitative review.

In [16]:
swap_target = VOTES_TARGETED[VOTES_TARGETED["voted_player_start_role"] != VOTES_TARGETED["voted_player_end_role"]].copy()
print(f"Swap-target votes (voted player's role changed): {len(swap_target)} / {len(VOTES_TARGETED)}")
print(swap_target.groupby(["model", "decoding_group"]).size().unstack(fill_value=0))

ROLE_ATTRIBUTION_PATTERN = re.compile(
    r"(?<!\w)(" + "|".join(re.escape(r) for r in ALL_ROLES) + r")(?!\w)", flags=re.IGNORECASE
)


def classify_swap_attribution(row):
    mentions = {m.group(1).title() for m in ROLE_ATTRIBUTION_PATTERN.finditer(row["justification"])}
    has_end = row["voted_player_end_role"] in mentions
    has_start = row["voted_player_start_role"] in mentions
    if has_end and has_start:
        return "both"
    if has_end:
        return "end_only"
    if has_start:
        return "start_only"
    if mentions:
        return "neither"
    return "no_mention"


swap_target["attribution"] = swap_target.apply(classify_swap_attribution, axis=1)
attribution_counts = swap_target.groupby(["model", "decoding_group", "attribution"]).size().unstack(fill_value=0)
print("\nRole-attribution classification on swap-target votes:")
print(attribution_counts)

# Confound check: is this just verbosity? Compare avg n_words across models (all targeted votes).
print("\nAvg n_words per justification (verbosity check):")
print(VOTES_TARGETED.groupby("model")["n_words"].mean().round(1))

Swap-target votes (voted player's role changed): 877 / 1963
decoding_group  Greedy  Stochastic
model                             
2B                  67         196
31B                 70         223
4B                  81         240



Role-attribution classification on swap-target votes:
attribution           both  end_only  neither  no_mention  start_only
model decoding_group                                                 
2B    Greedy             5        21       23           1          17
      Stochastic        13        60       59           0          64
31B   Greedy            32        12        5           1          20
      Stochastic        95        38       11           0          79
4B    Greedy            16        18       23           0          24
      Stochastic        47        57       54           2          80

Avg n_words per justification (verbosity check):
model
2B     63.6
31B    74.5
4B     81.8
Name: n_words, dtype: float64


In [17]:
swap_target["is_both"] = (swap_target["attribution"] == "both").astype(float)
per_game_both = (
    swap_target.groupby(["model", "decoding_group", "game_id"])
    .agg(share_both=("is_both", "mean"), n_instances=("is_both", "size"))
    .reset_index()
)
share_both_by_model = (
    per_game_both.groupby(["model", "decoding_group"])
    .agg(n_games=("game_id", "nunique"), n_instances=("n_instances", "sum"), mean_share_both=("share_both", "mean"))
    .reset_index()
)
print(share_both_by_model.round(3))


def unpaired_cross_model_ci(per_game_df, value_col, decoding_group, model_pairs=(("31B", "2B"), ("31B", "4B"), ("4B", "2B"))):
    sub = per_game_df[per_game_df["decoding_group"] == decoding_group]
    rows = []
    for m1, m2 in model_pairs:
        a = sub.loc[sub["model"] == m1, value_col].values
        b = sub.loc[sub["model"] == m2, value_col].values
        diff, lo, hi = boot_diff(a, b)
        rows.append({
            "decoding_group": decoding_group, "model_a": m1, "model_b": m2,
            "n_games_a": len(a), "n_games_b": len(b),
            "mean_a": round(float(a.mean()), 3) if len(a) else np.nan,
            "mean_b": round(float(b.mean()), 3) if len(b) else np.nan,
            "diff": round(diff, 3) if not np.isnan(diff) else np.nan,
            "ci_low": round(lo, 3) if not np.isnan(lo) else np.nan,
            "ci_high": round(hi, 3) if not np.isnan(hi) else np.nan,
            "excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
            "inconclusive_at_this_n": bool(np.isnan(lo) or not (lo > 0 or hi < 0)),
        })
    return pd.DataFrame(rows)


swap_target_bootstrap_ci = pd.concat(
    [unpaired_cross_model_ci(per_game_both, "share_both", dg) for dg in DECODING_ORDER], ignore_index=True
)
print("\nCross-model contrast on share_both (unpaired -- swap-target subset differs per model):")
print(swap_target_bootstrap_ci)

  model decoding_group  n_games  n_instances  mean_share_both
0    2B         Greedy       67           67            0.075
1    2B     Stochastic       90          196            0.061
2   31B         Greedy       70           70            0.457
3   31B     Stochastic       87          223            0.420
4    4B         Greedy       81           81            0.198
5    4B     Stochastic      103          240            0.194



Cross-model contrast on share_both (unpaired -- swap-target subset differs per model):
  decoding_group model_a model_b  n_games_a  n_games_b  mean_a  mean_b   diff  ci_low  ci_high  excludes_zero  inconclusive_at_this_n
0     Stochastic     31B      2B         87         90   0.420   0.061  0.358   0.262    0.456           True                   False
1     Stochastic     31B      4B         87        103   0.420   0.194  0.225   0.113    0.337           True                   False
2     Stochastic      4B      2B        103         90   0.194   0.061  0.133   0.060    0.210           True                   False
3         Greedy     31B      2B         70         67   0.457   0.075  0.383   0.251    0.512           True                   False
4         Greedy     31B      4B         70         81   0.457   0.198  0.260   0.110    0.405           True                   False
5         Greedy      4B      2B         81         67   0.198   0.075  0.123   0.014    0.232           Tru

### Precision layer -- DiMLex-connective check (reuses existing validated machinery)

Among `both` instances, does a DiMLex Temporal/Comparison marker (from the
already-built `DIMLEX_LEXICON`) sit textually between the start-role mention
and the end-role mention? Evidence the two mentions are linked by an
explicit contrast/temporal connective ("started as X **but** is now Y")
rather than being two disconnected observations in the same paragraph. No
new vocabulary is introduced -- this reuses the same published lexicon
already used in Analyses 1 and 2.

In [18]:
TEMPORAL_COMPARISON_CATS = {"Temporal", "Comparison"}


def connective_between_roles(text, start_role, end_role):
    start_matches = list(make_phrase_pattern(start_role).finditer(text))
    end_matches = list(make_phrase_pattern(end_role).finditer(text))
    if not start_matches or not end_matches:
        return False
    s, e = start_matches[0], end_matches[0]
    window = (s.end(), e.start()) if s.start() < e.start() else (e.end(), s.start())
    if window[0] >= window[1]:
        return False
    for m in find_matches(text, DIMLEX_LEXICON):
        if m["category"] in TEMPORAL_COMPARISON_CATS and m["start"] >= window[0] and m["end"] <= window[1]:
            return True
    return False


both_df = swap_target[swap_target["attribution"] == "both"].copy()
both_df["has_connective"] = both_df.apply(
    lambda r: connective_between_roles(r["justification"], r["voted_player_start_role"], r["voted_player_end_role"]),
    axis=1,
)
print(f"'both' instances: {len(both_df)}; with a linking Temporal/Comparison connective: {int(both_df['has_connective'].sum())}")

per_game_connective = both_df.groupby(["model", "decoding_group", "game_id"])["has_connective"].mean().reset_index()
connective_rows = []
for (mdl, dg), grp in per_game_connective.groupby(["model", "decoding_group"]):
    n_inst = int(both_df[(both_df["model"] == mdl) & (both_df["decoding_group"] == dg)].shape[0])
    mean, lo, hi = boot_mean(grp["has_connective"].values)
    connective_rows.append({
        "model": mdl, "decoding_group": dg, "n_games": len(grp), "n_instances": n_inst,
        "pct_with_connective": round(100 * mean, 1) if not np.isnan(mean) else np.nan,
        "ci_low": round(100 * lo, 1) if not np.isnan(lo) else np.nan,
        "ci_high": round(100 * hi, 1) if not np.isnan(hi) else np.nan,
    })
swap_narration_connective_check = pd.DataFrame(connective_rows)
print(swap_narration_connective_check)

'both' instances: 208; with a linking Temporal/Comparison connective: 120
  model decoding_group  n_games  n_instances  pct_with_connective  ci_low  ci_high
0    2B         Greedy        5            5                 60.0    20.0    100.0
1    2B     Stochastic       10           13                 50.0    20.0     80.0
2   31B         Greedy       32           32                 56.2    37.5     71.9
3   31B     Stochastic       48           95                 64.9    54.2     75.3
4    4B         Greedy       16           16                 56.2    31.2     81.2
5    4B     Stochastic       34           47                 63.2    48.0     78.4


# Save outputs and final sanity check

In [19]:
preexisting_files = {f.name for f in OUTPUT_DIR.glob("*.csv")}

outputs = {
    "04a_contingency_by_surrogate_predictability.csv": surrogate_means,
    "04a_contingency_by_surrogate_predictability_bootstrap_ci.csv": surrogate_contingency_ci,
    "04b_markers_by_herded_independent.csv": vote_class_counts.reset_index(),
    "04b_markers_by_herded_independent_bootstrap_ci.csv": markers_by_herded_independent_ci,
    "04b_excluded_other_votes.csv": excluded_other,
    "04c_rule_reference_lexicon_by_model.csv": rule_rates_by_model.reset_index(),
    "04c_rule_reference_lexicon_bootstrap_ci.csv": rule_reference_bootstrap_ci,
    "04c_rule_reference_lexicon_coverage.csv": rule_lexicon_coverage,
    "04d_mention_check_faithfulness.csv": mention_check,
    "05a_swap_target_role_attribution.csv": attribution_counts.reset_index(),
    "05a_swap_target_role_attribution_bootstrap_ci.csv": swap_target_bootstrap_ci,
    "05b_swap_narration_connective_check.csv": swap_narration_connective_check,
}

foreign_files = {f for f in preexisting_files if not f.startswith(("04a", "04b", "04c", "04d"))}
new_collisions = set(outputs) & foreign_files
assert not new_collisions, f"Filename collision with pre-existing DiMLex-extension outputs: {new_collisions}"

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")
    assert len(df) > 0, f"{name} is empty"

print("Saved (new):")
for name in outputs:
    print(" -", name)
print("\nPre-existing DiMLex-extension files in the same directory (untouched):")
for name in sorted(preexisting_files):
    print(" -", name)

Saved (new):
 - 04a_contingency_by_surrogate_predictability.csv
 - 04a_contingency_by_surrogate_predictability_bootstrap_ci.csv
 - 04b_markers_by_herded_independent.csv
 - 04b_markers_by_herded_independent_bootstrap_ci.csv
 - 04b_excluded_other_votes.csv
 - 04c_rule_reference_lexicon_by_model.csv
 - 04c_rule_reference_lexicon_bootstrap_ci.csv
 - 04c_rule_reference_lexicon_coverage.csv
 - 04d_mention_check_faithfulness.csv
 - 05a_swap_target_role_attribution.csv
 - 05a_swap_target_role_attribution_bootstrap_ci.csv
 - 05b_swap_narration_connective_check.csv

Pre-existing DiMLex-extension files in the same directory (untouched):
 - 04a_contingency_by_surrogate_predictability.csv
 - 04a_contingency_by_surrogate_predictability_bootstrap_ci.csv
 - 04b_excluded_other_votes.csv
 - 04b_markers_by_herded_independent.csv
 - 04b_markers_by_herded_independent_bootstrap_ci.csv
 - 04c_rule_reference_lexicon_bootstrap_ci.csv
 - 04c_rule_reference_lexicon_by_model.csv
 - 04c_rule_reference_lexicon_cove

## Summary: which contrasts are conclusive at this n, and the two judgment calls to review

- Every bootstrap table above carries `excludes_zero` / `inconclusive_at_this_n`
  per row -- only rows with `excludes_zero == True` should be narrated as a
  real difference in the write-up.
- **`excludes_zero` is necessary but not sufficient.** Cross-check Analysis 3
  rows against the coverage table: `night_order_mechanic` clears the CI bar in
  most cells while resting on ~5-12% of justifications *and* suffering the
  privileged-information confound described in that section. Report
  `role_name` (99% coverage) and `card_mechanic` (44%) as the findings;
  leave `night_order_mechanic` as exploratory.
- **Judgment call 1:** the mechanic-term list in Analysis 3
  (`CARD_MECHANIC_TERMS`, `NIGHT_ORDER_MECHANIC_TERMS`) is a starting
  proposal with reasoning per term; review before finalizing, especially the
  generic `night` term (compare `rate_night_order_mechanic` against its
  `_noNight` variant).
- **Judgment call 2:** `most_accused_player` ties are broken alphabetically
  (deterministic) rather than randomly, unlike the surrogate notebook's own
  baseline -- reasonable for a descriptive analysis, but worth knowing if
  results are compared directly against the surrogate's `most_accused_ww`
  baseline numbers.
- **Analysis 5 caveat:** `share_both` (mentions both the start and end role
  for a swap-target vote) is the intended headline number for "does the
  bigger model deduce rule-based swaps more" -- but co-occurrence is not
  proof of assertion. The DiMLex-connective check raises confidence when it
  also clears its own CI, but neither check certifies causal faithfulness.

In [20]:
excludes_zero_summary = pd.concat([
    surrogate_contingency_ci.assign(analysis="1_surrogate_predictability")[["analysis", "model", "decoding_group", "excludes_zero"]],
    markers_by_herded_independent_ci.assign(analysis="2_herded_vs_independent")[["analysis", "model", "decoding_group", "marker_category", "excludes_zero"]],
    rule_reference_bootstrap_ci.assign(analysis="3_rule_reference_cross_model")[["analysis", "decoding_group", "model_a", "model_b", "marker_category", "excludes_zero"]],
    swap_target_bootstrap_ci.assign(analysis="5_swap_target_role_attribution")[["analysis", "decoding_group", "model_a", "model_b", "excludes_zero"]],
], ignore_index=True)
print(excludes_zero_summary.to_string())

                          analysis model decoding_group  excludes_zero                    marker_category model_a model_b
0       1_surrogate_predictability    2B         Greedy           True                                NaN     NaN     NaN
1       1_surrogate_predictability    2B     Stochastic          False                                NaN     NaN     NaN
2       1_surrogate_predictability   31B         Greedy           True                                NaN     NaN     NaN
3       1_surrogate_predictability   31B     Stochastic          False                                NaN     NaN     NaN
4       1_surrogate_predictability    4B         Greedy          False                                NaN     NaN     NaN
5       1_surrogate_predictability    4B     Stochastic          False                                NaN     NaN     NaN
6          2_herded_vs_independent    2B         Greedy          False                        Contingency     NaN     NaN
7          2_herded_vs_i